## Expert Knowledge Worker

 ### A question answering agent that is an expert knowledge worker
 ### To be used by employees of Insurellm, an Insurance Tech company
 ### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

This first implementation will use a simple, brute-force type of RAG..

 ### Sidenote: Business applications of this week's projects

RAG is perhaps the most immediately applicable technique of anything that we cover in the course! In fact, there are commercial products that do precisely what we build this week: nuanced querying across large databases of information, such as company contracts or product specs. RAG gives you a quick-to-market, low cost mechanism for adapting an LLM to your business area.

In [1]:
# imports
import os
import glob
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI
import pprint as pp

In [2]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4o-mini"

In [3]:
# Load environment variables in a file called .env
load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
openai = OpenAI()

In [4]:
# With massive thanks to student Dr John S. for fixing a bug in the below for Windows users!
context = {}

employees = glob.glob("knowledge-base/employees/*")

for employee in employees:
    name = employee.split(' ')[-1][:-3]
    print(name)
    doc = ""
    with open(employee, "r", encoding="utf-8") as f:
        doc = f.read()
    context[name]=doc

Greene
Thomson
Carter
Trenton
Harper
Spencer
Lancaster
Chen
Bishop
Thompson
Blake
Tran


In [5]:
# print(context["Lancaster"])

In [88]:
products = glob.glob("knowledge-base/products/*")

for product in products:
    name = product.split(os.sep)[-1][:-3]
    doc = ""
    with open(product, "r", encoding="utf-8") as f:
        doc = f.read()
    context[name]=doc

In [6]:
context.keys()

dict_keys(['Greene', 'Thomson', 'Carter', 'Trenton', 'Harper', 'Spencer', 'Lancaster', 'Chen', 'Bishop', 'Thompson', 'Blake', 'Tran'])

In [7]:
system_message = """You are an expert in answering accurate questions about Insurellm, 
the Insurance Tech company. Give brief, accurate answers. 
If you don't know the answer, say so. Do not make anything up if you haven't been provided with relevant context."""

In [8]:
print(system_message)

You are an expert in answering accurate questions about Insurellm, 
the Insurance Tech company. Give brief, accurate answers. 
If you don't know the answer, say so. Do not make anything up if you haven't been provided with relevant context.


In [9]:
def get_relevant_context(message):
    relevant_context = []
    for context_title, context_details in context.items():
        if context_title.lower() in message.lower():
            print(f"get_relevant_context() : Context found for : {context_title.lower()} \n")
            relevant_context.append(context_details)
    return relevant_context          

In [10]:
def add_context(message):
    relevant_context = get_relevant_context(message)
    print(f" add_context(): Relevant context list length : {len(relevant_context)}  {type(relevant_context)}")
    # print(f" {relevant_context}")
    if relevant_context:
        message += "\n\nThe following additional context might be relevant in answering this question:\n\n"
        for relevant in relevant_context:
            message += relevant + "\n\n"

    return message

In [11]:
result = get_relevant_context("Who is lancaster?")
for i, r in enumerate(result):
    print(f"item {i} {'-'*88}\n{r}")

get_relevant_context() : Context found for : lancaster 

item 0 ----------------------------------------------------------------------------------------
# Avery Lancaster

## Summary
- **Date of Birth**: March 15, 1985  
- **Job Title**: Co-Founder & Chief Executive Officer (CEO)  
- **Location**: San Francisco, California  

## Insurellm Career Progression
- **2015 - Present**: Co-Founder & CEO  
  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  

- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  
  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the tech sector.  

- **2010 - 2013**: Business Anal

In [12]:
result = get_relevant_context("Who is Avery Lancaster and what is Carllm?")
print(len(result))
for i, r in enumerate(result):
    print(f"--- START item {i} {'-'*88}")
    print(f"{r}")
    print(f"--- END   item {i} {'-'*88}\n")

get_relevant_context() : Context found for : lancaster 

1
--- START item 0 ----------------------------------------------------------------------------------------
# Avery Lancaster

## Summary
- **Date of Birth**: March 15, 1985  
- **Job Title**: Co-Founder & Chief Executive Officer (CEO)  
- **Location**: San Francisco, California  

## Insurellm Career Progression
- **2015 - Present**: Co-Founder & CEO  
  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  

- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  
  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the tech sector.  

- **2010 - 2013**: B

In [13]:
result = add_context("Who is Alex Lancaster?")
print(len(result))
print(result)
# for i, r in enumerate(result):
#     print(f"--- START item {i} {'-'*88}")
#     print(f"{r}")
#     print(f"--- END   item {i} {'-'*88}\n")

get_relevant_context() : Context found for : lancaster 

 add_context(): Relevant context list length : 1  <class 'list'>
4437
Who is Alex Lancaster?

The following additional context might be relevant in answering this question:

# Avery Lancaster

## Summary
- **Date of Birth**: March 15, 1985  
- **Job Title**: Co-Founder & Chief Executive Officer (CEO)  
- **Location**: San Francisco, California  

## Insurellm Career Progression
- **2015 - Present**: Co-Founder & CEO  
  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  

- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  
  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking in

In [14]:
history = [""]
messages = [{"role": "system", "content": system_message}] + history
message = "Who is alex lancaster"
message = add_context(message)
# print(message)
# print('\n\n\n')
messages.append({"role": "user", "content": message})

pp.pprint(messages)

get_relevant_context() : Context found for : lancaster 

 add_context(): Relevant context list length : 1  <class 'list'>
[{'content': 'You are an expert in answering accurate questions about '
             'Insurellm, \n'
             'the Insurance Tech company. Give brief, accurate answers. \n'
             "If you don't know the answer, say so. Do not make anything up if "
             "you haven't been provided with relevant context.",
  'role': 'system'},
 '',
 {'content': 'Who is alex lancaster\n'
             '\n'
             'The following additional context might be relevant in answering '
             'this question:\n'
             '\n'
             '# Avery Lancaster\n'
             '\n'
             '## Summary\n'
             '- **Date of Birth**: March 15, 1985  \n'
             '- **Job Title**: Co-Founder & Chief Executive Officer (CEO)  \n'
             '- **Location**: San Francisco, California  \n'
             '\n'
             '## Insurellm Career Progression\n'

In [15]:
def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history
    message = add_context(message)
    messages.append({"role": "user", "content": message})
    print('\n\n\n')
    print('='*100)
    pp.pprint(messages)
    print('='*100)
    
    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

## Now we will bring this up in Gradio using the Chat interface -

A quick and easy way to prototype a chat with an LLM

In [16]:
view = gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


get_relevant_context() : Context found for : lancaster 

 add_context(): Relevant context list length : 1  <class 'list'>




[{'content': 'You are an expert in answering accurate questions about '
             'Insurellm, \n'
             'the Insurance Tech company. Give brief, accurate answers. \n'
             "If you don't know the answer, say so. Do not make anything up if "
             "you haven't been provided with relevant context.",
  'role': 'system'},
 {'content': 'Who is Avery Lancaster?\n'
             '\n'
             'The following additional context might be relevant in answering '
             'this question:\n'
             '\n'
             '# Avery Lancaster\n'
             '\n'
             '## Summary\n'
             '- **Date of Birth**: March 15, 1985  \n'
             '- **Job Title**: Co-Founder & Chief Executive Officer (CEO)  \n'
             '- **Location**: San Francisco, California  \n'
             '\n'
             '## Insurellm Career Progression\n